In [2]:
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.anthropic import AnthropicModel
from anthropic import Anthropic
import os

from rag_helper import RAGHelper
from ingest import load_faq_data, build_index
import json

In [3]:
load_dotenv()  # Load environment variables from .env file
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [5]:
model = AnthropicModel("claude-haiku-4-5-20251001")

In [ ]:
agent = Agent(model=model, system_prompt="You are a helpful assistant.")

In [10]:
documents = load_faq_data()
index = build_index(documents)

/home/drezende/.local/share/uv/python/cpython-3.14.4-linux-x86_64-gnu/lib/python3.14/ast.py:46: RuntimeWarning: coroutine 'AbstractAgent.run' was never awaited
  return compile(source, filename, mode, flags,


KeyboardInterrupt: 

In [ ]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use ONLY the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [ ]:
agent = Agent(model=model, system_prompt=instructions)

In [ ]:
# Register the search function as a tool using a decorator
@agent.tool_plain
def search_tool(query: str) -> list[dict]:
    """Search the FAQ database for entries matching the query."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [ ]:
result = await agent.run("How do I run Olama locally?")
print(result)

In [ ]:
result.all_messages()

In [ ]:
# A way to keep the conversation with the LLM
result2 = await agent.run(
    "How do I run a different model?",
    message_history=result.all_messages(),
)

print(result2)